In [16]:
import pandas as pd

df_gmv = pd.read_csv("./cleaned_data/gmv_clean_sales_infor.csv", encoding="utf-8")
df_leads = pd.read_csv("./cleaned_data/leads_clean_with_note_phone.csv", encoding="utf-8")

In [ ]:
df_leads.head()

In [18]:
import pandas as pd
import re

# =====================================================
# STEP 1. Normalize phone
# =====================================================

def normalize_phone(phone):
    if pd.isna(phone):
        return ""

    phone = str(phone).strip()

    # Bỏ .0
    if phone.endswith(".0"):
        phone = phone[:-2]

    # Chỉ giữ số
    phone = re.sub(r"\D", "", phone)

    return phone


# Chuẩn hóa phone
df_gmv["Phone_clean"] = df_gmv["Phone_clean"].apply(normalize_phone)

df_leads["Phone_clean"] = df_leads["Phone_clean"].apply(normalize_phone)

df_leads["Phone_extracted_from_note"] = (
    df_leads["Phone_extracted_from_note"]
    .apply(normalize_phone)
)

# Chuẩn hóa Sales
df_gmv["Sales_infor"] = (
    df_gmv["Sales_infor"]
    .fillna("")
    .astype(str)
    .str.strip()
)

df_leads["TVTS"] = (
    df_leads["TVTS"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# Thêm index để debug nếu cần
df_leads = (
    df_leads
    .reset_index(drop=False)
    .rename(columns={"index": "lead_index"})
)

# =====================================================
# STEP 2. Hàm chọn candidate tốt nhất
# =====================================================

def choose_best_match(candidates, sales):

    # Ưu tiên TVTS giống Sales_infor
    same_sales = candidates[candidates["TVTS"] == sales]

    if len(same_sales) > 0:
        return same_sales.iloc[0]

    # Nếu không có thì lấy dòng đầu tiên
    return candidates.iloc[0]


# =====================================================
# STEP 3. Matching
# =====================================================

results = []

# Những cột của GMV
gmv_cols = set(df_gmv.columns)

for _, gmv_row in df_gmv.iterrows():

    phone = gmv_row["Phone_clean"]
    sales = gmv_row["Sales_infor"]

    matched = None
    match_type = "No Match"

    # -----------------------------------
    # Match bằng Phone_clean
    # -----------------------------------

    if phone != "":

        candidates = df_leads[
            df_leads["Phone_clean"] == phone
        ]

        if len(candidates):

            matched = choose_best_match(
                candidates,
                sales
            )

            match_type = "Phone_clean"

    # -----------------------------------
    # Nếu chưa match thì dùng
    # Phone_extracted_from_note
    # -----------------------------------

    if matched is None and phone != "":

        candidates = df_leads[
            df_leads["Phone_extracted_from_note"] == phone
        ]

        if len(candidates):

            matched = choose_best_match(
                candidates,
                sales
            )

            match_type = "Phone_extracted_from_note"

    # -----------------------------------
    # Ghép dữ liệu
    # -----------------------------------

    row = gmv_row.to_dict()

    if matched is not None:

        for col in df_leads.columns:

            if col == "lead_index":
                continue

            # Nếu trùng tên cột thì thêm hậu tố _lead
            output_col = (
                f"{col}_lead"
                if col in gmv_cols
                else col
            )

            row[output_col] = matched[col]

    else:

        for col in df_leads.columns:

            if col == "lead_index":
                continue

            output_col = (
                f"{col}_lead"
                if col in gmv_cols
                else col
            )

            row[output_col] = pd.NA

    row["match_type"] = match_type

    results.append(row)


# =====================================================
# STEP 4. Kết quả
# =====================================================

final_df = pd.DataFrame(results)

print(final_df["match_type"].value_counts())


# =====================================================
# STEP 5. Lấy các dòng chưa match
# =====================================================

unmatched_gmv = final_df[
    final_df["match_type"] == "No Match"
].copy()

print(f"Unmatched: {len(unmatched_gmv)}")

match_type
Phone_clean                  280
No Match                     180
Phone_extracted_from_note      3
Name: count, dtype: int64
Unmatched: 180


In [19]:
unmatched_gmv = final_df.query("match_type == 'No Match'")

In [20]:
print(unmatched_gmv[[
    "UID_clean",
    "Phone_clean",
    "Sales_infor"
]].head(20))

     UID_clean    Phone_clean    Sales_infor
0   3311001921    84908224672   HCM - PhatNM
1   3179205818    84909517679  HCM - LinhLNT
3   3310902627    84389970625  HCM - LinhLNT
9   3311951330    84938222653  HCM - LinhLNT
10  3311919278    84844534222   HCM - PhatNM
11  3302992772    84909283440  HCM - LinhLNT
12  3311804423    84938350660   HCM - PhatNM
13  3312085874    84908026765  HCM - LinhLNT
14  3290550889    84944551229  HCM - LinhLNT
15  3312125506    84966027720  HCM - LinhLNT
17  3312266413    84908868028  HCM - LinhLNT
20  3309785398   817031882708   HN - TuyetLT
24  3307250645   818051986529   HN - TuyetLT
28  3310976565  4917644461422   HN - ThuyPTT
29  3310871114   491631303303  HN - TrinhVTT
30  3310314777    84978112361   HN - TrangNT
35  3311084949    84528228888     HN2 - LyVC
36  3310593198    84967669255     HN - MyLTH
37  3310871114   491631303303  HN - TrinhVTT
38  3287789579  4915205315239    HN - NgaNTH


In [21]:
unmatched_gmv.to_csv(
    "./cleaned_data/gmv_unmatched.csv",
    index=False,
    encoding="utf-8-sig"
)

In [22]:
matched_uid = set(
    final_df.loc[
        final_df["match_type"] != "No Match",
        "UID_clean"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
)

# Loại bỏ UID rỗng nếu có
matched_uid.discard("")

In [23]:
remaining_gmv = df_gmv[
    ~df_gmv["UID_clean"]
        .fillna("")
        .astype(str)
        .str.strip()
        .isin(matched_uid)
].copy()

print("Remaining rows:", len(remaining_gmv))
print("Remaining unique UID:", remaining_gmv["UID_clean"].nunique())

Remaining rows: 180
Remaining unique UID: 173


In [25]:
remaining_gmv["UID_clean"] = (
    remaining_gmv["UID_clean"]
    .fillna("")
    .astype(str)
    .str.strip()
)

df_leads["UID_clean"] = (
    df_leads["UID_clean"]
    .fillna("")
    .astype(str)
    .str.strip()
)

In [26]:
uid_not_found_df = remaining_gmv[
    ~remaining_gmv["UID_clean"].isin(df_leads["UID_clean"])
].copy()

print("Rows without UID in Leads:", len(uid_not_found_df))

Rows without UID in Leads: 180


In [27]:
gmv_uid = set(remaining_gmv["UID_clean"]) - {""}
lead_uid = set(df_leads["UID_clean"]) - {""}

intersection = gmv_uid & lead_uid

print("Common UID:", len(intersection))

Common UID: 0


In [32]:
df_unmatched = pd.read_csv("./cleaned_data/gmv_unmatched.csv", encoding='utf-8')


In [33]:
check_name = (
    df_unmatched.merge(
        df_leads[
            ["Họ và tên", "TVTS", "Phone_clean", "UID_clean"]
        ],
        left_on=["User Name", "Sales_infor"],
        right_on=["Họ và tên", "TVTS"],
        how="inner",
        suffixes=("_gmv", "_lead")
    )[
        [
            "User Name",
            "Sales_infor",
            "Họ và tên",
            "TVTS",
            "Phone_clean_gmv",
            "UID_clean_gmv",
            "Phone_clean_lead",
            "UID_clean_lead",
        ]
    ]
)

print(f"Số dòng match: {len(check_name)}")
check_name.head()

KeyError: "['Họ và tên', 'TVTS'] not in index"

In [35]:
check_name = df_unmatched.merge(
    df_leads[["Họ và tên", "TVTS", "Phone_clean", "UID_clean"]],
    left_on=["User Name", "Sales_infor"],
    right_on=["Họ và tên", "TVTS"],
    how="inner",
    suffixes=("_gmv", "_lead")
)

print(check_name.columns.tolist())

['bank day', 'bank time', 'Gateway', 'User Name', 'Phone', 'UID', 'Package', 'Fixed/ non-fixed', 'Pay Time', 'Real Pay(VND)', 'GMV (RMB)', 'Payment Method', 'Type', 'Sales', 'Note', 'Month of payment', 'Unnamed: 16', 'TEAM', 'Full Price(VND)', '采购价格 \r\n（套餐配置）', '渠道号', '首次申请试听日期', '首次申请试听渠道号', '首次购课时间', 'Source', '已批工单', '总 B (被推荐） 课数', 'A (推荐人） 课数 (送PH)', 'A', 'A (推荐人）课数 (送UK)', 'B', '实际卖课单价 VND', '实际卖课单价 RMB', 'PH/UK', '采购价格（包括转介绍赠送课)', 'Nguồn', 'UID_clean_gmv', 'Phone_clean_gmv', 'Sales_infor', 'Sales_Abbr', 'Đầu Ngày', 'Họ và tên_gmv', 'Người trực', 'Số điện thoại', 'Tuổi', 'Quốc Gia', 'Note_lead', 'TVTS_gmv', 'Ad_id', 'Nguồn_lead', 'Quốc gia \r\nQuảng cáo', 'Mẫu Quảng cáo', 'Mã CRM', 'Sale sau phân', 'Link FB', 'UID_lead', 'TT', 'Trạng thái', 'Ngày lên học thử', 'Current Binded Sale', 'Sales in latest system-weighted allocation', 'Latest system-weighted Allocation Time', 'Allocate reason', 'Tên sale chatpage', 'KOL', 'Check Ad_ID', 'UID_clean_lead', 'Phone_clean_lead', 'Phone_extr

In [36]:
check_name = check_name[
    [
        "User Name",
        "Sales_infor",
        "Họ và tên_lead",
        "TVTS_lead",
        "Phone_clean_gmv",
        "UID_clean_gmv",
        "Phone_clean_lead",
        "UID_clean_lead",
    ]
]

In [38]:
print(f"Số dòng match: {len(check_name)}")
check_name.head(8)

Số dòng match: 8


,User Name,Sales_infor,Họ và tên_lead,TVTS_lead,Phone_clean_gmv,UID_clean_gmv,Phone_clean_lead,Phone_clean_lead,UID_clean_lead,UID_clean_lead
0,Linh,HN - NgocNTT,Linh,HN - NgocNTT,84944822873,3311285843,NaN,817084769177,NaN,3310574905
1,Bảo Ngọc,HN - ThaoPT,Bảo Ngọc,HN - ThaoPT,84966778912,3311376451,NaN,821072898177,NaN,3309225968
2,Giang,HN2 - VanLTT,Giang,HN2 - VanLTT,84369771267,3311354979,NaN,84931133606,NaN,3311046085
3,Ngọc Hoa,HN - NhungLT,Ngọc Hoa,HN - NhungLT,84344213009,3311366847,NaN,84987683755,NaN,3310380159
4,Hồng Anh,HN - ThaoPT,Hồng Anh,HN - ThaoPT,84941538625,3311649925,NaN,817083975696,NaN,3306191423
5,Khang,HN - QuynhKTT,Khang,HN - QuynhKTT,84966155489,3311508463,NaN,819092683838,NaN,3307039875
6,Anh Thư,HN - AnhNT,Anh Thư,HN - AnhNT,84987189666,3304138773,NaN,84986265378,NaN,
7,Ngọc Anh,HN - TrangNT,Ngọc Anh,HN - TrangNT,84376521708,3312150282,NaN,84327515426,NaN,3307896674
